In [2]:
import requests
import pandas as pd
from datetime import datetime
from azure.storage.blob import BlobServiceClient
from io import BytesIO
from azure.core.exceptions import ResourceExistsError

StatementMeta(, 4148cc8f-fb24-43e2-afc8-c438dfacac15, 4, Finished, Available, Finished, False)

### Read key vault

In [3]:
BLOB_CONNECTION_STRING = mssparkutils.credentials.getSecret('https://champdemo4802355165.vault.azure.net/','storageKey')
CONTAINER_NAME = mssparkutils.credentials.getSecret('https://champdemo4802355165.vault.azure.net/', 'container')

StatementMeta(, 4148cc8f-fb24-43e2-afc8-c438dfacac15, 5, Finished, Available, Finished, False)

In [4]:
# Connect to Azure Blob

blob_service_client = BlobServiceClient.from_connection_string(
    BLOB_CONNECTION_STRING
)

container_client = blob_service_client.get_container_client(CONTAINER_NAME)

try:
    container_client.create_container()
except ResourceExistsError:
    pass

StatementMeta(, 4148cc8f-fb24-43e2-afc8-c438dfacac15, 6, Finished, Available, Finished, False)

In [5]:
# Helper function: fetch API data

def fetch_data(endpoint):

    url = f"https://api.openf1.org/v1/{endpoint}?session_key=latest"

    response = requests.get(url)

    if response.status_code != 200:
        print("API error:", response.status_code)
        return None

    data = response.json()

    if isinstance(data, list) and len(data) > 0:
        return pd.DataFrame(data)
    else:
        print(f"No data returned for {endpoint}")
        return None


StatementMeta(, 4148cc8f-fb24-43e2-afc8-c438dfacac15, 7, Finished, Available, Finished, False)

In [6]:
# convert dataframes to parqut files 

def upload_dataframe(df, name):
    
    # Generate timestamped parquet blob name
    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    file_name = f"{name}_{timestamp}.parquet"

    # Convert DataFrame to Parquet in memory
    buffer = BytesIO()
    # engine='pyarrow' (recommended) or 'fastparquet'
    df.to_parquet(buffer, index=False, engine="pyarrow", compression="snappy")
    buffer.seek(0)

    # Get a blob client and upload
    blob_client = blob_service_client.get_blob_client(
        container=CONTAINER_NAME,
        blob=file_name
    )
    blob_client.upload_blob(buffer, overwrite=True)

    print(f"Uploaded {file_name}")

StatementMeta(, 4148cc8f-fb24-43e2-afc8-c438dfacac15, 8, Finished, Available, Finished, False)

In [7]:
# Fetch endpoints

print("Fetching latest F1 session data...")

drivers_df = fetch_data("drivers")
laps_df = fetch_data("laps")
results_df = fetch_data("session_result")

StatementMeta(, 4148cc8f-fb24-43e2-afc8-c438dfacac15, 9, Finished, Available, Finished, False)

Fetching latest F1 session data...
API error: 401
API error: 401
API error: 401


In [8]:
def clean_gap(value):
    if pd.isna(value):
        return None
    
    value = str(value)
    
    if "LAP" in value:
        return None  # or use a large value like 9999
    
    try:
        return float(value)
    except:
        return None

# results_df["gap_to_leader"] = results_df["gap_to_leader"].apply(clean_gap)

if results_df is not None:
    results_df["gap_to_leader"] = results_df["gap_to_leader"].apply(clean_gap)
    upload_dataframe(results_df, "race_results")

StatementMeta(, 4148cc8f-fb24-43e2-afc8-c438dfacac15, 10, Finished, Available, Finished, False)

In [9]:
# Upload to Blob

if drivers_df is not None:
    upload_dataframe(drivers_df, "drivers")

if laps_df is not None:
    upload_dataframe(laps_df, "laps")

if results_df is not None:
    upload_dataframe(results_df, "race_results")


print("Extraction completed.")

StatementMeta(, 4148cc8f-fb24-43e2-afc8-c438dfacac15, 11, Finished, Available, Finished, False)

Extraction completed.
